In [54]:
import glob
import os
import numpy as np
import pandas as pd

# import seaborn as sns
import plotly.graph_objects as go


import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# Coleta dos dados

In [55]:
results_flows_directories = glob.glob("../../results/results_flows*/*")
results_flows_directories

['../../results/results_flows/kuririnPPO_s_50_p_4_a_1.0_c_0',
 '../../results/results_flows/greedyb_s_50_p_4_a_1.0_c_0']

In [56]:
metricas_de_coleta = [
    "tempo",
    "cpu_utilization",
    "bandwidth_utilization",
    "success",
    "latency",
    "duration",
    "running_sfcs",
    "cpu_saved",
    "shared_vnfs",
    "server_crashed",
]

In [57]:
big_data = pd.DataFrame()


def colect_data_from_alg_directory(results_flows_directories):
    data_nla = []

    for alg_dir in results_flows_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split("/")[-1].split("_")
        alg_name = simu_exec_name[0].split("\\")[0]
        reliability = float(simu_exec_name[-1])

        files = os.listdir(alg_dir)
        print(f"Simulação:{alg_name},{reliability}")
        print("Quantidade de csv: ", len(files))

        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
            except:
                continue

            primeiro_tempo = simulation_df["timestamp"].values[0]
            simulation_df["tempo"] = simulation_df[["timestamp"]].applymap(
                lambda x: x - primeiro_tempo
            )
            simulation_is_success = "sfc_cache_p4_50" in simulation_df["sfc_id"].values

            if simulation_is_success:
                simulacoes_okays = simulacoes_okays + 1
                simulation_df = simulation_df[metricas_de_coleta]

                # Arrendondar tempo
                simulation_df["tempo"] = simulation_df["tempo"].astype(int)

                simulation_df = simulation_df.replace("None", pd.NA)

                latency_col = simulation_df[["tempo", "latency"]]
                latency_col.dropna(inplace=True)
                latency_col.loc[:, "latency"] = latency_col["latency"].astype(float)

                ###############################################
                cumulative_sum_success = 0
                cumulative_avg_success = []
                for i, value in enumerate(simulation_df["success"]):
                    cumulative_sum_success += value
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df["success"] = cumulative_avg_success
                ###############################################

                simulation_df = simulation_df.groupby("tempo", as_index=False).mean(
                    numeric_only=True
                )
                latency_df = (
                    latency_col.groupby("tempo", as_index=False)
                    .mean(numeric_only=True)
                    .reset_index()
                )

                tempo_range = simulation_df["tempo"].max()
                df_mean = simulation_df.set_index("tempo").reindex(range(tempo_range + 1))
                df_mean = df_mean.fillna(method="ffill")
                df_mean = df_mean.reset_index()

                latency_df = latency_df.set_index("tempo").reindex(range(tempo_range + 1))
                latency_df = latency_df.fillna(method="ffill")
                latency_df = latency_df.reset_index()

                df_mean["latency"] = latency_df["latency"]
                df_mean["algorithm"] = alg_name
                df_mean["reliability"] = reliability
                # df_mean['sharing'] = share
                df_mean = df_mean.iloc[0:1000]
                data_nla.append(df_mean)

        data_nla_f = pd.concat(data_nla)
        print("Simulações de sucesso: ", simulacoes_okays)
        print("Dados Nulos: ", data_nla_f.isnull().sum().sum())
        print()

    return data_nla_f

In [61]:
big_data = colect_data_from_alg_directory(results_flows_directories)

Simulação:kuririnPPO,0.0
Quantidade de csv:  5


KeyError: "['duration', 'server_crashed'] not in index"

In [ ]:
big_data

,tempo,cpu_utilization,bandwidth_utilization,success,latency,duration,running_sfcs,cpu_saved,shared_vnfs,algorithm,reliability
0,0,0.031820,0.012840,1.0,2.200000,0.171390,3.0,5.000000,2.20,ga,0.95
1,1,0.037000,0.015225,1.0,2.000000,0.191701,3.0,2.083333,1.25,ga,0.95
2,2,0.039820,0.012020,1.0,1.600000,0.182153,2.8,0.000000,0.40,ga,0.95
3,3,0.081825,0.023800,1.0,0.500000,0.184759,6.5,5.333333,4.00,ga,0.95
4,4,0.083633,0.019267,1.0,1.333333,0.176280,6.0,0.000000,3.00,ga,0.95
...,...,...,...,...,...,...,...,...,...,...,...
995,995,0.584550,0.089525,1.0,0.750000,0.000249,46.5,75.000000,46.50,osfem,0.99
996,996,0.591050,0.089500,1.0,1.000000,0.000483,47.5,83.333333,50.50,osfem,0.99
997,997,0.591050,0.089500,1.0,1.000000,0.000483,47.5,83.333333,50.50,osfem,0.99
998,998,0.591050,0.089500,1.0,1.000000,0.000483,47.5,83.333333,50.50,osfem,0.99


In [ ]:
big_data.reliability.unique()

array([0.95 , 0.975, 0.99 ])

In [ ]:
big_data.reliability.unique()

In [ ]:
big_data["CPU Salva por SFC"] = big_data["cpu_saved"] / big_data["running_sfcs"]
big_data["CPU Salva por Servidor"] = big_data["cpu_saved"] / 35

In [ ]:
# Suposições
# capacidade_maxima_banda_gbps = 10  # Capacidade máxima da banda em Gbps

# Calculando métricas
# big_data["eficiencia de cpu"] =  big_data["cpu_utilization"] / big_data["running_sfcs"]

# big_data["eficiencia de banda"] =   big_data["bandwidth_utilization"]   / big_data["running_sfcs"]
# big_data["eficiencia de cache"] =  big_data["cache_utilization"]  / big_data["running_sfcs"]

# big_data["bit_rate"] = big_data["practical_bandwidth_utilization"] * capacidade_maxima_banda_gbps

# # Função para calcular a pontuação da latência
# def calcular_pontuacao_latencia(latencia):
#     if pd.isna(latencia):
#         return 0  # Latência Nula
#     elif latencia > 6:
#         return -1  # Latência Ruim
#     else:
#         return 2  # Latência Boa

# # Função para calcular a pontuação da aceitação
# def calcular_pontuacao_success(success):
#     # Convertendo a taxa de sucesso para uma escala de 0 a 1 e multiplicando por 10 para obter uma pontuação máxima de 10
#     return success * 10

# # Aplicando as funções para calcular as pontuações
# big_data['pontuacao_latencia'] = big_data['Latency'].apply(calcular_pontuacao_latencia)
# big_data['pontuacao_success'] = big_data['success'].apply(calcular_pontuacao_success)

# # Calculando a métrica final de qualidade do serviço
# big_data['QoS'] = big_data['pontuacao_latencia'] + big_data['pontuacao_success']

In [ ]:
# Lista de algoritmos a serem analisados
algoritmos = ["gr", "msf", "musfico"]

# Dicionário para armazenar os dados processados de cada algoritmo
dados_processados = {}


def process_data(data):
    data = data.groupby("tempo").mean()
    return data


for alg in algoritmos:
    # Filtrando os dados baseado no algoritmo e na condição de compartilhamento
    dados_filtrados = big_data[(big_data["algorithm"] == alg) & (big_data["sharing"] == "4")]

    # Removendo as colunas 'algor}ithm' e 'sharing'
    dados_filtrados = dados_filtrados.drop(["algorithm", "sharing"], axis=1)

    # Processando os dados filtrados
    dados_processados[alg] = process_data(dados_filtrados)

In [ ]:
dados_processados

{'gr':        cpu_utilization  bandwidth_utilization   success   latency  duration  \
 tempo                                                                         
 0             0.044237               0.014674  1.000000  1.748781  0.023046   
 1             0.053428               0.004674  1.000000  0.387051  0.019252   
 2             0.058163               0.010666  1.000000  1.488910  0.021430   
 3             0.067615               0.012534  1.000000  1.352068  0.021161   
 4             0.067615               0.012534  1.000000  1.352068  0.021161   
 ...                ...                    ...       ...       ...       ...   
 995           0.558789               0.134878  0.985243  1.771930  0.022411   
 996           0.562058               0.136232  0.985254  1.855263  0.022657   
 997           0.561656               0.135727  0.985246  1.903509  0.023733   
 998           0.556714               0.131942  0.985237  1.526316  0.024218   
 999           0.554935           

In [ ]:
# Agora, dados_processados contém os dados processados para cada algoritmo
# Acessando os dados processados para cada algoritmo:
gr_data_share = dados_processados["gr"]
msf_data_share = dados_processados["msf"]
musfico_data_share = dados_processados["musfico"]

In [ ]:
print(msf_data_share["duration"].mean())
print(musfico_data_share["duration"].mean())
print(gr_data_share["duration"].mean())

0.23444108547697462
0.23235482277619207
0.025214901812078604


# Plot de linha

In [ ]:
x = gr_data_share.index.values

In [ ]:
metricas = {
    "Taxa de Aceitação (%)": "success",
    "CPU (%)": "cpu_utilization",
    "Largura de Banda (%)": "bandwidth_utilization",
    "Latência (ms)": "latency",
    "Tempo de Decisão(s)": "duration",
    "SF's Compartilhadas": "shared_vnfs",
    # "Bit Rate (Gbps)": "bit_rate",
    # "Quality of Service (QoS)": "QoS",
    # "Running SFC's":"running_sfcs"
}


# Nomes dos algoritmos para legendas
legendas = {"gr": "OSFEM", "msf": "MSF", "musfico": "MuSFiCO"}

In [ ]:
import pandas as pd


def plotly_three_lines_graph_with_error_bars(
    y2,
    y3,
    y4,
    xaxis_title="Tempo",
    yaxis_title="Y Axis",
    linha2="linha2",
    linha3="linha3",
    linha4="linha4",
    steps=1,
    x_scale_factor=100,
):
    # Calcular média
    y2_mean = y2.groupby(np.arange(len(y2)) // steps).mean()
    y3_mean = y3.groupby(np.arange(len(y3)) // steps).mean()
    y4_mean = y4.groupby(np.arange(len(y4)) // steps).mean()

    # Calcular desvio padrão
    y2_std = y2.groupby(np.arange(len(y2)) // steps).std()
    y3_std = y3.groupby(np.arange(len(y3)) // steps).std()
    y4_std = y4.groupby(np.arange(len(y4)) // steps).std()

    # Criar índices para o eixo X e aplicar transformação de escala
    index = np.arange(
        0, len(y2), steps
    )  # / x_scale_factor  # Transformação de escala aplicada aqui

    # Ajustar título do eixo X para refletir a reescalação
    scale_info = " (s)"
    adjusted_xaxis_title = xaxis_title + scale_info

    # Plot
    fig = go.Figure()

    # Adicionar linhas e barras de erro
    fig.add_trace(
        go.Scatter(
            x=index,
            y=y2_mean,
            mode="lines+markers",
            name=linha2,
            line=dict(dash="solid", color="#E61C83"),  # Azul brilhante
            error_y=dict(type="data", array=y2_std, visible=True),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=index,
            y=y3_mean,
            mode="lines+markers",
            name=linha3,
            line=dict(dash="solid", color="#1CCEE6"),  # Roxo
            error_y=dict(type="data", array=y3_std, visible=True),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=index,
            y=y4_mean,
            mode="lines+markers",
            name=linha4,
            line=dict(dash="solid", color="#E6CF1B"),  # Laranja
            error_y=dict(type="data", array=y4_std, visible=True),
        )
    )

    # Definição dos pontos específicos e labels

    # Adição das linhas tracejadas e labels
    # Encontrar o valor máximo entre todas as médias para definir um fim lógico para as linhas tracejadas
    max(y2_mean.max(), y3_mean.max(), y4_mean.max())

    # for x, label in zip(specific_x_values, labels):
    #     # Usar max_y_value * algum fator (por exemplo, 1.1) para garantir que as linhas se estendam além dos pontos mais altos
    #     fig.add_shape(type="line", x0=x, y0=0, x1=x, y1=max_y_value * 1.1, line=dict(dash="dash", color="grey"))
    #     fig.add_annotation(x=x, y=max_y_value * 1.1, text=label, showarrow=True, arrowhead=0)

    # Atualizar layout do gráfico
    fig.update_layout(
        yaxis=dict(
            showline=True, showgrid=True, title=yaxis_title, gridcolor="lightgray", gridwidth=2
        ),
        xaxis=dict(
            showline=True,
            showgrid=True,
            title=adjusted_xaxis_title,
            gridcolor="lightgray",
            gridwidth=2,
        ),
        legend_title=None,
        margin=dict(l=120, r=10, b=100, t=25),
        autosize=True,
        width=700,
        height=600,
        template="plotly_white",
        legend=dict(x=0.1, y=1.14, traceorder="normal", orientation="h", itemwidth=30),
        font=dict(family="Arial", size=35, color="Black"),
    )

    # if yaxis_title == "Taxa de Aceitação (%)":
    #     fig.update_layout(
    #         yaxis=dict(
    #             range=[75, 100],  # Set the y-axis range
    #             showgrid=True,
    #             title=yaxis_title,
    #             gridcolor='lightgray',
    #             gridwidth=2
    #         ),
    #     )

    title = yaxis_title.split(" ")[0] + ".pdf"
    fig.write_image(title)
    fig.show()  # Uncomment this line if you want to display the plot in an interactive environment

In [ ]:
# Iterando sobre cada métrica para plotar
dados_alg = []
dados_plotagem = []
for titulo, coluna in metricas.items():
    # Multiplicar por 100 quando necessário para converter em porcentagem
    multiplicador = 100 if "%" in titulo else 1

    # Preparando dados para plotagem
    dados_plotagem = []
    for alg in ["gr", "msf", "musfico"]:
        dados_alg = dados_processados[alg][coluna] * multiplicador
        dados_plotagem.append(dados_alg)

    # Chamada para a função de plotagem com os dados preparados
    plotly_three_lines_graph_with_error_bars(
        *dados_plotagem,
        yaxis_title=titulo,
        linha2=legendas["gr"],
        linha3=legendas["msf"],
        linha4=legendas["musfico"],
        steps=100,
    )

In [ ]:
# percentage_metrics = [
#     "success",
#     "bandwidth_utilization",
#     "cpu_utilization",
#     "cache_utilization"
# ]

# factor = 100  # Factor by which to multiply the values

# # Iterate over each algorithm (key) and its DataFrame (value) in the dictionary
# for algorithm, df in dados_processados.items():
#     # Check if the DataFrame contains the columns you're interested in
#     cols_to_multiply = [col for col in percentage_metrics if col in df.columns]
#     # Multiply the specified columns by the factor
#     df[cols_to_multiply] = df[cols_to_multiply].apply(lambda x: x * factor)

In [ ]:
dados_processados["gr"]

,cpu_utilization,bandwidth_utilization,success,latency,duration,running_sfcs,cpu_saved,shared_vnfs,CPU Salva por SFC,CPU Salva por Servidor
tempo,,,,,,,,,,
0,0.044237,0.014674,1.000000,1.748781,0.023046,4.142032,9.085751,5.322612,2.151062,0.259593
1,0.053428,0.004674,1.000000,0.387051,0.019252,4.803968,9.648688,6.119653,1.996768,0.275677
2,0.058163,0.010666,1.000000,1.488910,0.021430,5.339098,11.537437,6.772274,2.194279,0.329641
3,0.067615,0.012534,1.000000,1.352068,0.021161,6.275251,15.063440,8.524123,2.372365,0.430384
4,0.067615,0.012534,1.000000,1.352068,0.021161,6.275251,15.063440,8.524123,2.372365,0.430384
...,...,...,...,...,...,...,...,...,...,...
995,0.558789,0.134878,0.985243,1.771930,0.022411,50.868421,117.109649,63.587719,2.342736,3.345990
996,0.562058,0.136232,0.985254,1.855263,0.022657,51.320175,118.900585,64.535088,2.358139,3.397160
997,0.561656,0.135727,0.985246,1.903509,0.023733,51.311404,119.371345,64.640351,2.367689,3.410610


In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import numpy as np

# def plot_metrics_in_subplots(y_data, xaxis_title='Time (s)', metrics_list=[{'Metric 1': 'm1', 'Metric 2': 'm2', 'Metric 3': 'm3'}], legends={'alg1': 'Algorithm 1', 'alg2': 'Algorithm 2', 'alg3': 'Algorithm 3'}, steps=1, font_size=14, titles=["title1", "title2"], layout_options=None):
#     if layout_options is None:
#         layout_options = {}

#     layout_config = {
#         'height': 900,
#         'width': 1200,
#         'legend_x': 1,
#         'legend_y': 1,
#         'legend_xanchor': 'right',
#         'legend_yanchor': 'top',
#         'legend_border': 1.1
#     }

#     layout_config.update(layout_options)

#     fig = make_subplots(rows=len(metrics_list), cols=len(metrics_list[0]), horizontal_spacing=0.11, vertical_spacing=0.13)

#     colors = ['#EF553B', '#00CC96', '#FFA15A']
#     dashes = ['dash', 'dot', 'solid']

#     added_legend = set()

#     for row, metrics in enumerate(metrics_list, 1):
#         for col, (metric_name, metric_key) in enumerate(metrics.items(), 1):
#             for alg, color, dash in zip(legends.keys(), colors, dashes):
#                 y = y_data[alg][metric_key]
#                 y = y.groupby(np.arange(len(y))//steps).mean()
#                 x = np.arange(len(y)) * steps

#                 show_legend = alg not in added_legend
#                 added_legend.add(alg)

#                 fig.add_trace(
#                     go.Scatter(x=x, y=y, mode='lines', name=legends[alg] if show_legend else None, line=dict(color=color, dash=dash), showlegend=show_legend),
#                     row=row, col=col
#                 )

#     fig.update_layout(
#         height=layout_config['height'],
#         width=layout_config['width'],
#         showlegend=True,
#         legend_tracegroupgap=50,
#         legend=dict(
#             x=layout_config['legend_x'],
#             y=layout_config['legend_y'],
#             xanchor=layout_config['legend_xanchor'],
#             yanchor=layout_config['legend_yanchor'],
#             borderwidth=layout_config['legend_border'],
#             orientation='h',
#             itemsizing="constant", itemwidth=70
#          ),
#         template="plotly_white",
#         margin=dict(l=40, r=40, t=40, b=40),
#         font=dict(size=font_size)
#     )

#     tick_values = [0, 200, 400, 600, 800, 1000]
#     for row, metrics in enumerate(metrics_list, 1):
#         for col, metric_name in enumerate(metrics.keys(), 1):
#             tickformat = ".1f" if metric_name == "Decision Time (s)" else ".1f"
#             tickformat = "d" if metric_name == "Bandwidth Utilization (%)" or metric_name == "CPU Utilization (%)"or metric_name == "Cache Utilization (%)" or metric_name ==  "Acceptance Ratio (%)" else ".1f"
#             fig.update_xaxes(title_text=xaxis_title, row=row, col=col, tickvals=tick_values, ticktext=[str(value) for value in tick_values], range=[0, 1000], title_font=dict(size=font_size), tickfont=dict(size=font_size))
#             fig.update_yaxes(title_text=metric_name, row=row, col=col, title_font=dict(size=font_size), tickfont=dict(size=font_size), tickformat=tickformat)

#     fig.show()

#     title = titles[0].split(" ")[0] + ".pdf"
#     fig.write_image(title)


# legendas = {
#      'msf': 'MSF       ',
#      'musfico': 'MusFiCO         ',
#      'gr': 'MasCo'
# }

# # Exemplo de uso da função ajustada com os dados específicos fornecidos
# metricas_1 = {
#     "Acceptance Ratio (%)": "success",
#     "Decision Time (s)": "duration",
#     "Latency (s)": "Latency"
# }

# # Exemplo de uso da função ajustada com os dados específicos fornecidos
# metricas_2 = {
#     "Bandwidth Utilization (%)": "bandwidth_utilization",
#     "CPU Utilization (%)": "cpu_utilization",
#     "Cache Utilization (%)": "cache_utilization"
# }


# # Define layout options for customization
# layout_options = {
#     'height': 900,  # Example: Change plot height
#     'width': 1600,   # Example: Change plot width
#     'legend_x':0.5,  # Keep legend on the right
#     'legend_y': 1.1,  # Keep legend at the top
#     'legend_xanchor': 'center',  # Anchor legend to the right side
#     'legend_yanchor': 'top',    # Anchor legend to the top
#     'legend_border':2,
#     'legend_width':130,
# }

# # Call the function with the data and layout options
# plot_metrics_in_subplots(
#     y_data=dados_processados,
#     xaxis_title='Time (s)',
#     metrics_list=[metricas_1, metricas_2],
#     legends=legendas,
#     steps=10,
#     font_size=30,  # Adjust font size if needed
#     titles=["Network Metrics", "Resource Utilization"],
#     layout_options=layout_options  # Pass the layout options here
# )


# Gráficos BoxPlot

In [ ]:
# import plotly.express as px

# def plotly_boxplot_graph(y2, y3,y4,xaxis_title='Tempo (s)', yaxis_title='Y Axis', linha2='linha2', linha3='linha3',linha4='linha4',steps=200):
#     y2_f = pd.DataFrame(y2)
#     y3_f = pd.DataFrame(y3)
#     y4_f = pd.DataFrame(y4)
#     # y5_f = pd.DataFrame(y5)

#     index = list(range(steps,1000+steps,steps))

#     index_col = []
#     for i in index:
#         index_col.extend([i]*steps)

#     y2_f['Time(s)'] = index_col
#     y3_f['Time(s)'] = index_col
#     y4_f['Time(s)'] = index_col
#     # y5_f['Time(s)'] = index_col

#     y2_f['alg'] = linha2
#     y3_f["alg"] = linha3
#     y4_f["alg"] = linha4
#     # y5_f["alg"] = linha5

#     # df = pd.concat([y2_f,y3_f,y4_f,y5_f],axis=0)
#     df = pd.concat([y2_f,y3_f,y4_f],axis=0)
#     df.columns = [yaxis_title,xaxis_title,"alg"]

#     pio.templates["draft"] = go.layout.Template(
#         layout_annotations=[
#             dict(
#                 xref="paper",
#                 yref="paper",
#                 showarrow=True,
#             )
#         ]
#     )
#     fig = px.box(df, x=df.columns[1], y=df.columns[0],color="alg")
#     fig.update_traces(quartilemethod="exclusive")

#     fig.update_layout(
#         yaxis=dict(showline=True,showgrid=True),
#         xaxis=dict(showline=True,showgrid=True),
#         legend_title=None,
#         margin=dict(l=100, r=10, b=80, t=25),
#         autosize=True,
#         width=950,
#         height=600,
#         template="draft",
#         legend=dict(x=0.05, y=1.2, traceorder='normal', orientation='h'),
#         font=dict(
#         family="Arial",  # Especifique o tipo de fonte desejado
#         size=25,  # Especifique o tamanho da fonte desejado
#         color="Black"  # Especifique a cor da fonte desejada
#     )
#     )
#     fig.show()
#     title = yaxis_title.split(" ")[0] + ".pdf"
#     #fig.write_image(title)

In [ ]:
# # Dicionário com os títulos das métricas e os nomes das colunas correspondentes
# metricas = {
#     "CPU Utilization(%)": "practical_cpu_utilization",
#     "Cache Utilization(%)": "practical_cache_utilization",
#     "Bandwidth Utilization(%)": "practical_bandwidth_utilization",
#     "Latency (ms)": "Latency",
#     "Acceptance Ratio (%)": "success",
#     "Decision Time (s)": "duration"
# }

# # Nomes dos algoritmos para legendas
# legendas = {
#     'rodrigo': 'Guided Cost',
#     'ga': 'Genetic Algorithm',
#     'dp': 'Dynamic Programming'
# }

# # Iterando sobre cada métrica para plotagem
# for titulo, coluna in metricas.items():
#     # Determinar se é necessário converter os valores para porcentagem
#     multiplicador = 100 if "%" in titulo else 1

#     # Preparando dados para a plotagem
#     dados_plotagem = []
#     for alg in ['msf', 'ga', 'gr']:
#         dados_alg = dados_processados[alg][coluna] * multiplicador
#         dados_plotagem.append(dados_alg)

#     # Chamada à função de plotagem com os dados preparados
#     plotly_boxplot_graph(*dados_plotagem,
#                          yaxis_title=titulo,
#                          linha2=legendas['msf'],
#                          linha3=legendas['ga'],
#                          linha4=legendas['gr'])
